# FACULTY SOLUTION — Week 3: Run rates and the shutdown point
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['refinery_costs', 'rotterdam_restart', 'window1_params', 'cohort_states', 'crack_history', 'jv_governance', 'worked_example_plant']}

In [ ]:

def compute(d):
    rc = d['refinery_costs'].set_index('refinery'); w = {x.parameter: x.value for x in d['window1_params'].itertuples()}
    r = {}
    for name, key in [('Baton Rouge', 'br'), ('Rotterdam', 'rot'), ('Singapore', 'sing')]:
        x = rc.loc[name]
        r[key + '_contribution'] = x.crack + x.complexity - x.variable_cost
        r[key + '_net'] = r[key + '_contribution'] - x.fixed_cost
        r[key + '_shutdown_crack'] = x.variable_cost - x.complexity
    r['rot_idle_delta'] = -r['rot_contribution'] + rc.loc['Rotterdam', 'avoidable_fixed']
    for s in d['cohort_states'].itertuples():
        crack = w['base_crack'] - w['slope'] * (s.util - w['base_util'])
        r['crack_' + s.state] = crack
        r['rot_net_' + s.state] = crack + rc.loc['Rotterdam', 'complexity'] - rc.loc['Rotterdam', 'variable_cost'] - rc.loc['Rotterdam', 'fixed_cost']
        r['rot_contribution_' + s.state] = crack + rc.loc['Rotterdam', 'complexity'] - rc.loc['Rotterdam', 'variable_cost']
    r['floor'] = w['shutdown_floor']
    return r


In [ ]:
fx = json.load(open('../fixtures/week3_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')